# ColabUMA Opt — geometry optimization with UMA

Optimizes a molecular structure with Meta's machine learning potential **UMA**, then runs a **DFT single point** ([GPU4PySCF](https://github.com/pyscf/gpu4pyscf)) at the optimized structure. No coding needed.

## How to use

1. In **Runtime → Change runtime type**, choose **T4 GPU**
2. First time only: open 🔑 (Secrets) in the left sidebar, add your Hugging Face access token with the name `HF_TOKEN`, and turn on notebook access
3. In **1. Input form** below, set the output name, charge, spin multiplicity, convergence and DFT settings
4. Choose **Runtime → Run all**
5. A file selection button appears in **2. File selection and checks**. Choose an xyz or a Gaussian gjf / com file. You can leave it running after that
6. Check the 3D view, energies, convergence plot and DFT energy in the cells below, and download the ZIP of results in the last cell

Only the structure is read from a gjf; the charge and multiplicity come from the form. `_opt.log` in the ZIP writes the optimization path in Gaussian's output format, so GaussView shows each step's structure and the energy profile (it is not a Gaussian calculation). The DFT energy is in `_dft.csv`.

To run again, repeat steps 3–6 (the install and the model load are skipped). In the file selection, Cancel upload reuses the previous file.

> **Before you start**: the model weights, [facebook/UMA](https://huggingface.co/facebook/UMA), require an access request. Log in to Hugging Face, accept the FAIR Chemistry License v1 on that page, and once approved, create a token with Read access under [Access Tokens](https://huggingface.co/settings/tokens).

## Notes

- An optimization can converge to a saddle point. Turn on the **vibrational analysis** to check important structures.
- UMA is a machine learning potential, not a replacement for DFT. Check final energy comparisons with DFT.
- For small molecules (up to a few tens of atoms), using DFT directly is often more reliable.
- The DFT single point uses the UMA structure as it is; the structure is not reoptimized with DFT. Open shells use UKS.
- DFT runs on GPU4PySCF (density fitting). The T4 is slow at double precision, so large molecules and def2-TZVPD take time. Without a GPU, DFT runs on the CPU with PySCF.
- Periodic systems (crystals, surfaces) are not supported.
- For scans of a bond length, angle or dihedral, use [ColabUMA_Scan](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Scan_en.ipynb); for transition state searches, use [ColabUMA_TS](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_TS_en.ipynb). To go from optimization to scan to TS in one session, use [ColabUMA_Workflow](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Workflow_en.ipynb): the install and the model load happen only once.

## License

- This notebook and the shared code (`colabuma`): [MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- UMA model weights: [FAIR Chemistry License v1](https://huggingface.co/facebook/UMA)
- Libraries installed at runtime are used under their own licenses. See [License in the README](https://github.com/RChibaChiba/ColabChem/blob/main/README.md#license)

## Citation

If you use this notebook in a publication, please cite:

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

If you use the DFT single point, please also cite:

- PySCF (Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020))
- GPU4PySCF (X. Wu *et al.*, arXiv:2404.09452 (2024))

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### Settings
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown Convergence threshold fmax (eV/Å). normal is close to Gaussian's default, tight to Gaussian's opt=tight.
convergence = "loose (0.05)"  #@param ["loose (0.05)", "normal (0.02)", "tight (0.001)", "custom"]
#@markdown fmax (eV/Å) used when custom is chosen.
fmax_custom = 0.05  #@param {type:"number"}
max_steps = 500  #@param {type:"integer"}
run_vibrations = False  #@param {type:"boolean"}

#@markdown ### DFT single point (gpu4pyscf)
#@markdown Computes the DFT energy at the optimized structure. ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
run_dft = False  #@param {type:"boolean"}
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]
#@markdown Time limit for DFT (minutes, all structures together). Past it, DFT is skipped and the UMA results are saved. 0 means no limit (DFT still stops before the Colab runtime is likely to end).
dft_time_limit = 60  #@param {type:"integer"}

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection and the token check happen here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, opt, output, structure, ui
from colabuma.ui import ColabUMAError, ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
if run_dft:
    dft.parse_method(dft_functional, dft_basis)
    if dft_time_limit < 0:
        raise ColabUMAError("The DFT time limit must be 0 or more.", "0 means no limit.",
                            ja="DFT の時間の上限は0以上にしてください。", ja_hint="0 なら上限なしです。")
env.check_gpu()
env.ensure_token()
upload_name, upload_data = structure.upload()
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install and login) { display-mode: "form" }
DEVICE = env.setup(["dft"] if run_dft else [])

In [ ]:
#@title 4. Load the model { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. Prepare the structures { display-mode: "form" }
atoms0, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity)
ok(f"Initial structure ready: {atoms0.get_chemical_formula()} ({len(atoms0)} atoms).")
if run_dft:
    dft.check_memory([("", atoms0)], charge, multiplicity, dft.parse_method(dft_functional, dft_basis)[1])

In [ ]:
#@title 6. Optimization { display-mode: "form" }
#@markdown The Converge bar shows how far fmax has come, in orders of magnitude, from its first value toward the threshold. It moves back if fmax rises.
print(f"Optimizing with {optimizer_name} (fmax < {fmax} eV/Å, at most {max_steps} steps)")
best = opt.optimize(atoms0, calc, job, "Structure")
vib_result = opt.vibrations(best["atoms"], calc) if run_vibrations else None

In [ ]:
#@title 7. Results { display-mode: "form" }
files, log = opt.report(best, vib_result, job)

In [ ]:
#@title 8. DFT single points { display-mode: "form" }
dft_log = []
if run_dft:
    _f, dft_log = dft.run_optional([("Optimized structure", best["atoms"])], charge, multiplicity, dft_functional, dft_basis,
                                   None, dft_time_limit)
    files.update(_f)
else:
    print("DFT single points are off.")

In [ ]:
#@title 9. Save { display-mode: "form" }
files["log.txt"] = output.log_text("ColabUMA Opt log", source_desc, job, [log, dft_log])
zip_path = output.save_zip(job.out_prefix, files)